In [5]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,mean_absolute_percentage_error
from xgboost import XGBRegressor

In [ ]:
# 1. Load data
df = pd.read_csv(r"train.csv")
df.head()


,Component1_fraction,Component2_fraction,Component3_fraction,Component4_fraction,Component5_fraction,Component1_Property1,Component2_Property1,Component3_Property1,Component4_Property1,Component5_Property1,...,BlendProperty1,BlendProperty2,BlendProperty3,BlendProperty4,BlendProperty5,BlendProperty6,BlendProperty7,BlendProperty8,BlendProperty9,BlendProperty10
0,0.21,0.00,0.42,0.25,0.12,-0.021782,1.981251,0.020036,0.140315,1.032029,...,0.489143,0.607589,0.321670,-1.236055,1.601132,1.384662,0.305850,0.193460,0.580374,-0.762738
1,0.02,0.33,0.19,0.46,0.00,-0.224339,1.148036,-1.107840,0.149533,-0.354000,...,-1.257481,-1.475283,-0.437385,-1.402911,0.147941,-1.143244,-0.439171,-1.379041,-1.280989,-0.503625
2,0.08,0.08,0.18,0.50,0.16,0.457763,0.242591,-0.922492,0.908213,0.972003,...,1.784349,0.450467,0.622687,1.375614,-0.428790,1.161616,0.601289,0.872950,0.660000,2.024576
3,0.25,0.42,0.00,0.07,0.26,-0.577734,-0.930826,0.815284,0.447514,0.455717,...,-0.066422,0.483730,-1.865442,-0.046295,-0.163820,-0.209693,-1.840566,0.300293,-0.351336,-1.551914
4,0.26,0.16,0.08,0.50,0.00,0.120415,0.666268,-0.626934,2.725357,0.392259,...,-0.118913,-1.172398,0.301785,-1.787407,-0.493361,-0.528049,0.286344,-0.265192,0.430513,0.735073


In [4]:
X = df.iloc[:, :55]    # component fraction + 10 properties
y = df.iloc[:, 55:]    # blend properties
y = y.apply(pd.to_numeric, errors='coerce').fillna(0)

In [7]:
# 3. Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [9]:
# 4. Train-validation split
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [11]:
# 5. Train model with XGBoost
xgb_model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
)
xgb_model.fit(X_train, y_train)

MultiOutputRegressor(estimator=XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.8, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.05, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=8,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=500, n_jobs=-1,
                                            num_parallel_tree=None, ...))

In [13]:
# 6. Validation predictions
y_pred_val = xgb_model.predict(X_val)

In [15]:
# 7. Evaluation Metrics
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)
mape = mean_absolute_percentage_error(y_val, y_pred_val)

print("===== Overall Validation Performance (XGBoost) =====")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape*100:.2f}%")

===== Overall Validation Performance (XGBoost) =====
RMSE : 0.3134
MAE  : 0.2320
R²   : 0.8996
MAPE : 260.15%


In [17]:
print("\n===== Per-Property Performance =====")
for i, col in enumerate(y.columns):
    rmse_i = np.sqrt(mean_squared_error(y_val.iloc[:, i], y_pred_val[:, i]))
    mae_i = mean_absolute_error(y_val.iloc[:, i], y_pred_val[:, i])
    r2_i = r2_score(y_val.iloc[:, i], y_pred_val[:, i])
    mape_i = mean_absolute_percentage_error(y_val.iloc[:, i], y_pred_val[:, i])
    print(f"{col}: RMSE={rmse_i:.4f}, MAE={mae_i:.4f}, R²={r2_i:.4f}, MAPE={mape_i*100:.2f}%")


===== Per-Property Performance =====
BlendProperty1: RMSE=0.3347, MAE=0.2613, R²=0.8782, MAPE=1283.20%
BlendProperty2: RMSE=0.2961, MAE=0.2289, R²=0.9047, MAPE=143.38%
BlendProperty3: RMSE=0.2637, MAE=0.2046, R²=0.9274, MAPE=100.88%
BlendProperty4: RMSE=0.3465, MAE=0.2674, R²=0.8781, MAPE=100.10%
BlendProperty5: RMSE=0.1391, MAE=0.0765, R²=0.9819, MAPE=29.70%
BlendProperty6: RMSE=0.3625, MAE=0.2765, R²=0.8670, MAPE=204.32%
BlendProperty7: RMSE=0.2674, MAE=0.2067, R²=0.9254, MAPE=96.34%
BlendProperty8: RMSE=0.3697, MAE=0.2865, R²=0.8581, MAPE=353.79%
BlendProperty9: RMSE=0.3755, MAE=0.2799, R²=0.8662, MAPE=222.36%
BlendProperty10: RMSE=0.3047, MAE=0.2313, R²=0.9093, MAPE=67.41%


In [21]:
# 8. Retrain on full data & predict test
xgb_model.fit(X_scaled, y)
y_test_pred = xgb_model.predict(X_val)

In [ ]:
import pickle

# assume your trained model is called "model"
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)
